# 03: Travel time during the June 2025 heat event

**Objective.** Determine whether trips recorded during the June 23 through 25 heat event show a
material difference in travel time within Brooklyn. The result determines whether these dates are
retained in the travel time fit after their exclusion from the demand fit.

**Context.** Notebook 02 identifies higher recorded trip volume during the event. Demand and travel
time are fitted separately, so the dates can be excluded from one input without being excluded from
the other.

## Analysis design

| Element | Specification |
|---|---|
| Analysis unit | Pickup zone, dropoff zone, and hour cell |
| Event dates | Monday, June 23 through Wednesday, June 25 |
| Reference dates | Other Mondays, Tuesdays, and Wednesdays in June |
| Main metric | Median log trip duration within each cell |
| Supporting metric | Median realized speed within each cell |
| Comparison | Median percentage difference across matched cells |
| Materiality threshold | 3% in either direction |

Grouping trips by pickup zone, dropoff zone, and hour reduces differences in route and time of day
between the event and reference samples. Exact trip distance still varies within a cell, so realized
speed provides a supporting measure of travel conditions.

The analysis also reports an overnight comparison and a sensitivity analysis that excludes June 2
and June 3 from the reference dates. The overnight period had a smaller demand increase than the
daytime period, although it was not unaffected by the event.

The bootstrap interval is calculated by resampling cells. It describes uncertainty across the
observed cells and does not account for dependence among nearby zones or adjacent hours. The
Wilcoxon statistic measures whether the distribution of cell differences is centered away from
zero. The artifact decision is based mainly on effect size relative to the 3% threshold.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from common import session

HOT = ("2025-06-23", "2025-06-24", "2025-06-25")  # Mon, Tue, Wed
CTRL = (
    "2025-06-02", "2025-06-09", "2025-06-16", "2025-06-30",  # Mondays
    "2025-06-03", "2025-06-10", "2025-06-17",                # Tuesdays
    "2025-06-04", "2025-06-11", "2025-06-18",                # Wednesdays
)
QUIET_START = ("2025-06-02", "2025-06-03")  # dropped in the sensitivity run

DAY_HOURS = tuple(range(9, 20))    # 09:00-19:00, where the demand anomaly sits
NIGHT_HOURS = (0, 1, 2, 3, 4, 5)   # overnight comparison

MIN_TRIPS = 30    # per cell, per group
MATERIAL = 0.03   # selected materiality threshold

con = session("2025-06")

## 1. Recorded volume on the event dates

The first calculation compares average Brooklyn pickup volume on the event and reference dates.
It confirms that the higher volume identified in Notebook 02 remains visible after restricting the
analysis to Brooklyn. This calculation uses the same trip records and is not independent evidence
of the event.

In [2]:
hot_sql = ", ".join(f"DATE '{d}'" for d in HOT)
ctrl_sql = ", ".join(f"DATE '{d}'" for d in CTRL)

vol = con.execute(f"""
    WITH bk AS (SELECT LocationID FROM z WHERE Borough = 'Brooklyn')
    SELECT CASE WHEN CAST(request_datetime AS DATE) IN ({hot_sql}) THEN 'hot'
                ELSE 'ctrl' END AS grp,
           CASE WHEN EXTRACT(hour FROM request_datetime) BETWEEN 9 AND 19
                THEN 'day' ELSE 'night' END AS part,
           COUNT(*) / COUNT(DISTINCT CAST(request_datetime AS DATE)) AS trips_per_day
    FROM t
    WHERE PULocationID IN (SELECT LocationID FROM bk)
      AND CAST(request_datetime AS DATE) IN ({hot_sql}, {ctrl_sql})
    GROUP BY 1, 2 ORDER BY 2, 1
""").df()

p = vol.pivot(index="part", columns="grp", values="trips_per_day")
p["pct"] = 100 * (p["hot"] / p["ctrl"] - 1)
p.round(1)

grp,ctrl,hot,pct
part,,,
day,86173.6,103526.7,20.1
night,66917.7,70721.3,5.7


## 2. Cell construction

The travel time comparison applies the following restrictions:

1. Both trip endpoints are in Brooklyn.
2. Matched shared trips are excluded because their durations can include a detour for another
   passenger.
3. Trip duration is between 60 seconds and two hours.
4. Each pickup zone, dropoff zone, and hour cell contains at least 30 trips in both the event and
   comparison samples.

The cell definition controls for the recorded taxi zones and hour. It does not hold exact distance
or route constant within a cell. Median realized speed is therefore reported with median duration.

In [3]:
def cell_stats(hours, ctrl_days):
    """Per (PU, DO, hour) cell: median log duration, dispersion, median speed, by group."""
    hot = ", ".join(f"DATE '{d}'" for d in HOT)
    ctrl = ", ".join(f"DATE '{d}'" for d in ctrl_days)
    hrs = ", ".join(str(h) for h in hours)
    return con.execute(f"""
        WITH bk AS (SELECT LocationID FROM z WHERE Borough = 'Brooklyn'),
        trips AS (
            SELECT PULocationID AS pu,
                   DOLocationID AS do_,
                   EXTRACT(hour FROM pickup_datetime) AS h,
                   CASE WHEN CAST(pickup_datetime AS DATE) IN ({hot})  THEN 'hot'
                        WHEN CAST(pickup_datetime AS DATE) IN ({ctrl}) THEN 'ctrl'
                   END AS grp,
                   LN(trip_time) AS lt,
                   trip_miles / (trip_time / 3600.0) AS mph
            FROM t
            WHERE PULocationID IN (SELECT LocationID FROM bk)
              AND DOLocationID IN (SELECT LocationID FROM bk)
              AND shared_match_flag = 'N'
              AND trip_time BETWEEN 60 AND 7200
              AND EXTRACT(hour FROM pickup_datetime) IN ({hrs})
              AND CAST(pickup_datetime AS DATE) IN ({hot}, {ctrl})
        )
        SELECT pu, do_, h, grp,
               COUNT(*) AS n,
               MEDIAN(lt) AS med_lt,
               STDDEV_SAMP(lt) AS sd_lt,
               QUANTILE_CONT(lt, 0.9) - QUANTILE_CONT(lt, 0.1) AS spread_lt,
               MEDIAN(mph) AS med_mph
        FROM trips WHERE grp IS NOT NULL
        GROUP BY pu, do_, h, grp
    """).df()


def paired(df):
    """Cells observed with >= MIN_TRIPS in both groups, widened to one row per cell."""
    w = df.pivot_table(index=["pu", "do_", "h"], columns="grp",
                       values=["n", "med_lt", "sd_lt", "spread_lt", "med_mph"])
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    w = w.dropna()
    return w[(w["n_hot"] >= MIN_TRIPS) & (w["n_ctrl"] >= MIN_TRIPS)]


def boot_ci(x, reps=10_000, seed=20250623):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(reps, len(x)))
    return np.percentile(np.median(x[idx], axis=1), [2.5, 97.5])


def report(w, label):
    d = (w["med_lt_hot"] - w["med_lt_ctrl"]).to_numpy()   # log ratio
    pct = 100 * (np.exp(d) - 1)
    lo, hi = boot_ci(pct)
    res = stats.wilcoxon(d, alternative="two-sided", zero_method="wilcox")
    print(f"\n--- {label} ---")
    print(f"cells paired            : {len(w):,}")
    print(f"trips (hot / ctrl)      : {int(w['n_hot'].sum()):,} / {int(w['n_ctrl'].sum()):,}")
    print(f"median duration shift   : {np.median(pct):+.2f}%   95% CI [{lo:+.2f}, {hi:+.2f}]")
    print(f"IQR of cell shifts      : [{np.percentile(pct, 25):+.2f}, {np.percentile(pct, 75):+.2f}]")
    print(f"cells slower on hot days: {100 * (pct > 0).mean():.1f}%")
    print(f"Wilcoxon signed-rank    : W={res.statistic:.0f}, p={res.pvalue:.3g}")
    print(f"median d(sd log dur)    : {np.median((w['sd_lt_hot'] - w['sd_lt_ctrl']).to_numpy()):+.4f}")
    print(f"median d(p90-p10 log)   : {np.median((w['spread_lt_hot'] - w['spread_lt_ctrl']).to_numpy()):+.4f}")
    print(f"median speed shift      : {100 * np.median(w['med_mph_hot'] / w['med_mph_ctrl'] - 1):+.2f}%")
    status = "WITHIN THRESHOLD" if max(abs(lo), abs(hi)) < 100 * MATERIAL else "OUTSIDE THRESHOLD"
    print(f"vs +/-{100 * MATERIAL:.0f}% threshold    : {status} (cell bootstrap interval)")
    return {"run": label, "cells": len(w), "median_pct": np.median(pct),
            "ci_lo": lo, "ci_hi": hi, "p": res.pvalue}

## 3. Comparisons

The main comparison covers 09:00 through 19:00, when the recorded volume increase is largest.
The overnight comparison covers 00:00 through 05:00. Recorded volume was 5.7% higher overnight,
so this comparison provides a lower exposure reference rather than an unaffected baseline.

The sensitivity analysis repeats the daytime comparison after removing June 2 and June 3 from the
comparison sample. This tests whether the main result depends on two relatively low volume dates
at the beginning of the month.

In [4]:
out = [
    report(paired(cell_stats(DAY_HOURS, CTRL)), "MAIN COMPARISON  daytime 09-19"),
    report(paired(cell_stats(NIGHT_HOURS, CTRL)), "OVERNIGHT COMPARISON  00-05"),
    report(paired(cell_stats(DAY_HOURS, tuple(d for d in CTRL if d not in QUIET_START))),
           "SENSITIVITY  daytime, early month dates dropped"),
]
pd.DataFrame(out)


--- MAIN COMPARISON  daytime 09-19 ---
cells paired            : 2,026
trips (hot / ctrl)      : 121,495 / 304,737
median duration shift   : -1.45%   95% CI [-1.99, -0.92]
IQR of cell shifts      : [-7.22, +5.02]
cells slower on hot days: 44.0%
Wilcoxon signed-rank    : W=865880, p=2.38e-09
median d(sd log dur)    : +0.0035
median d(p90-p10 log)   : -0.0042
median speed shift      : -1.04%
vs +/-3% threshold    : WITHIN THRESHOLD (cell bootstrap interval)

--- OVERNIGHT COMPARISON  00-05 ---
cells paired            : 97
trips (hot / ctrl)      : 4,678 / 14,465
median duration shift   : -0.20%   95% CI [-1.58, +1.05]
IQR of cell shifts      : [-4.71, +3.98]
cells slower on hot days: 49.5%
Wilcoxon signed-rank    : W=2235, p=0.611
median d(sd log dur)    : -0.0119
median d(p90-p10 log)   : -0.0400
median speed shift      : +0.36%
vs +/-3% threshold    : WITHIN THRESHOLD (cell bootstrap interval)



--- SENSITIVITY  daytime, early month dates dropped ---
cells paired            : 2,022
trips (hot / ctrl)      : 121,362 / 256,130
median duration shift   : -2.11%   95% CI [-2.62, -1.47]
IQR of cell shifts      : [-8.10, +4.41]
cells slower on hot days: 41.9%
Wilcoxon signed-rank    : W=785308, p=2.13e-19
median d(sd log dur)    : +0.0032
median d(p90-p10 log)   : -0.0026
median speed shift      : -0.21%
vs +/-3% threshold    : WITHIN THRESHOLD (cell bootstrap interval)


,run,cells,median_pct,ci_lo,ci_hi,p
0,MAIN COMPARISON daytime 09-19,2026,-1.454682,-1.989068,-0.921924,2.376542e-09
1,OVERNIGHT COMPARISON 00-05,97,-0.204290,-1.578509,1.048218,6.106470e-01
2,"SENSITIVITY daytime, early month dates dropped",2022,-2.111070,-2.615923,-1.465355,2.126783e-19


## 4. Results

| Comparison | Median duration difference | Cell bootstrap interval | Median speed difference |
|---|---:|---:|---:|
| Daytime | −1.45% | [−1.99%, −0.92%] | −1.04% |
| Overnight | −0.20% | [−1.58%, 1.05%] | 0.36% |
| Daytime sensitivity | −2.11% | [−2.62%, −1.47%] | −0.21% |

The daytime duration differences and their cell bootstrap intervals remain within the selected 3%
materiality threshold. The realized speed differences are also smaller than 3%. Changes in the
within cell standard deviation and percentile spread of log duration are small in both daytime
analyses.

For this event and sample, the analysis does not identify a material change in Brooklyn travel
time under the selected threshold. This result applies to observed completed trips during these
three dates. It does not establish that heat has no effect on travel conditions generally.

### Use in the fitted inputs

| Fitted input | Use of June 23 through 25 | Basis |
|---|---|---|
| Demand rates | Excluded | Recorded daytime volume increased by 20.1% in Brooklyn |
| Travel time distributions | Retained | Duration and speed differences remained within the 3% threshold |

This separate handling preserves the observed demand event for scenario construction while
retaining the event dates in the travel time sample. This notebook documents that existing fitting
decision and does not modify either fitted input.

### Limitations

- The event sample contains three dates from one heat event.
- The analysis is observational and cannot attribute the measured differences to temperature alone.
- Taxi zone pairs do not fix exact trip distance or route.
- The bootstrap treats cells as independent sampling units.
- The comparison sample contains only three or four dates for each day of the week.
- The overnight comparison includes a 5.7% increase in recorded volume and is not an unaffected baseline.
- The trip records contain completed trips, so changes in unserved or canceled requests are not
  observed.

## 5. Retained modeling tradeoffs

| Choice | Reason retained | Consequence |
|---|---|---|
| Retain the heat event dates in the travel fit | Removing three dates would reduce support in sparse origin, destination, and hour cells, while the estimated duration and speed changes remain below the selected 3 percent threshold. | Small event specific differences are incorporated into the baseline travel distributions. |
| Exclude the same dates from the demand fit | Recorded daytime volume increased materially during the event, so the baseline demand rates should not average the event into ordinary dates. | The baseline fit contains fewer dates. The event must be introduced separately when disruption performance is evaluated. |
| Require at least 30 trips in both samples | The threshold reduces instability in cell medians and permits matched comparisons. | Sparse routes and hours do not contribute to the result, so the conclusion is strongest for common movements. |
| Compare taxi zone pair and hour cells | Conditioning reduces differences in geography and time of day between the samples. | Exact route and distance remain uncontrolled within a cell. Changes in route mix can therefore appear as travel time changes. |
| Apply a 3 percent materiality threshold | The threshold separates small fitted differences from changes considered large enough to justify a separate travel treatment. | The value is a modeling criterion rather than an estimated operational threshold. Effects below 3 percent may still matter near capacity or charging constraints. |
| Use completed trips only | The analysis directly measures realized duration and speed. | It cannot measure canceled or unserved requests, or travel conditions for movements that did not produce a completed trip. |

The retained decision favors cell coverage in the travel artifact. Its consequence is that the
baseline includes any event related travel effect smaller than the selected threshold.